In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [3]:
class MLP(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias = False)
        self.up_proj = nn.Linear(d_model, d_ff, bias = False)
        self.down_proj = nn.Linear(d_ff, d_model, bias = False)

    def forward(self, x):
        gate = self.gate_proj(x)
        up = self.up_proj(x)
        fused = F.silu(gate) * up
        output = self.down_proj(fused)
        return output

In [4]:
torch.manual_seed(0)
d_model, d_ff, seq_len = 8, 16, 5

mlp = MLP(d_model, d_ff)
x = torch.rand(seq_len, d_model)

output = mlp(x)
print(output.shape)  # should be (seq_len, d_model) = (5, 8)

torch.Size([5, 8])


In [6]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = (x / torch.sqrt(torch.mean(x**2, dim = -1, keepdim=True) + self.eps))
        output = rms * self.weight
        return output


In [7]:
torch.manual_seed(0)
d_model, seq_len = 8, 5

norm = RMSNorm(d_model)
x = torch.rand(seq_len, d_model)

output = norm(x)
print(output.shape)  # should be (5, 8)

torch.Size([5, 8])
